# 03 — Real-conversation intent-tree extraction (Morpheus smoke test)

This notebook is the **real-conversation intent-tree extraction** — the thesis method: build a
DiscoverLLM-style intent tree from a real **ThoughtTrace** conversation via Morpheus (Qwen), apply
the **regrouping** step, and check it against a frozen Claude **Sonnet** extraction of the same
conversation (a reliability check on the extractor). The structural comparison of real-conversation
trees vs. DiscoverLLM's artifact-derived trees lives in **NB4**.

> ⏱️ **Morpheus is slow.** A single extraction can take many minutes (the rebuild notes report
> 1–60+ min and frequent empty/timeout responses). Run this **interactively / overnight** — batch
> execution (e.g. `nbconvert`) tends to hit a cell timeout.

> ⚠️ **Validity.** The Sonnet-vs-Morpheus comparison is valid only against fixed, frozen prompt
> versions, pinned and re-hashed below:
> - extraction `intent_tree_extraction.md` — v1.1 — `f1924f822892`
> - regrouping `intent_tree_regrouping.md` — v1 — `33ed398829ff`

**Steps:** (a) connectivity call · (b) extraction · (b2) always-on regrouping · (c) structural comparison vs. the frozen Sonnet extraction.

## Setup (credentials from `.env` — never hardcoded)

Reads from a git-ignored `.env` at the repo root: `MORPHEUS_KEY` (**required**), optional
`MORPHEUS_MODEL`, `MORPHEUS_BASE_URL`, `TEMPERATURE`, `TARGET_CONV_ID` (default
`user1016_task1_conversation1`), `HF_TOKEN`.

Bundled inputs:
- `prompts/intent_tree_extraction.md` — frozen v1.1 prompt (`f1924f822892`)
- `prompts/intent_tree_regrouping.md` — frozen v1 regrouping protocol (`33ed398829ff`)
- `data/sonnet_extractions/user1016_task1_conversation1.json` — frozen Sonnet extraction

**Morpheus must be streamed** — a non-streaming call returns nginx `504` for long generations.

In [ ]:
import os
import json
import time
import hashlib
from collections import Counter
from pathlib import Path

import pandas as pd
from openai import OpenAI
from datasets import load_dataset
from dotenv import load_dotenv

def _repo_root():
    # Marker independent of the (removed) reproduction package.
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "requirements.txt").exists() and (cand / "notebooks").is_dir():
            return cand
    return Path.cwd()

ROOT = _repo_root()
load_dotenv(ROOT / ".env")   # MORPHEUS_KEY etc. (git-ignored); never hardcode secrets

MORPHEUS_BASE_URL = os.environ.get("MORPHEUS_BASE_URL", "https://morpheus.cit.tum.de/api/")
MORPHEUS_MODEL = os.environ.get("MORPHEUS_MODEL", "Qwen/Qwen3.6-35B-A3B")
TEMPERATURE = float(os.environ.get("TEMPERATURE", "0.2"))
MORPHEUS_KEY = os.environ.get("MORPHEUS_KEY")
if not MORPHEUS_KEY:
    raise RuntimeError("Set MORPHEUS_KEY in your .env at the repo root. Never hardcode it.")

client = OpenAI(base_url=MORPHEUS_BASE_URL, api_key=MORPHEUS_KEY)
print(f"Morpheus endpoint: {MORPHEUS_BASE_URL}")
print(f"Model           : {MORPHEUS_MODEL}")

In [ ]:
# Morpheus REQUIRES streaming (non-stream -> nginx 504). Retry on the two known transient
# failures: an empty stream on a 200 OK (server-side model failure) and ReadTimeout.
def stream_chat(messages, *, json_mode=False, retries=3, timeout=600):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            kwargs = dict(model=MORPHEUS_MODEL, messages=messages,
                          temperature=TEMPERATURE, stream=True, timeout=timeout)
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            stream = client.chat.completions.create(**kwargs)
            parts = []
            for chunk in stream:
                delta = chunk.choices[0].delta.content if chunk.choices else None
                if delta:
                    parts.append(delta)
            text = "".join(parts)
            if not text.strip():
                raise RuntimeError("empty stream (server-side model failure)")
            return text
        except Exception as err:  # smoke test: keep error handling simple
            last_err = err
            print(f"  attempt {attempt}/{retries} failed: {err!r}")
            time.sleep(2 * attempt)
    raise RuntimeError(f"Morpheus call failed after {retries} attempts: {last_err!r}")

# (a) One successful chat-completion call.
reply = stream_chat([
    {"role": "system", "content": "You are a terse connectivity probe."},
    {"role": "user", "content": "Reply with exactly one short sentence confirming you are reachable."},
])
print("Morpheus reply:\n", reply)

In [ ]:
# Load + verify BOTH frozen prompts (hash-gated; comparison validity depends on fixed versions).
EXTRACTION_PROMPT_PATH = Path(os.environ.get("EXTRACTION_PROMPT_PATH", ROOT / "prompts" / "intent_tree_extraction.md"))
REGROUP_PROMPT_PATH = Path(os.environ.get("REGROUP_PROMPT_PATH", ROOT / "prompts" / "intent_tree_regrouping.md"))
EXPECTED_EXTRACTION_HASH = "f1924f822892"   # intent_tree_extraction.md v1.1
EXPECTED_REGROUP_HASH = "33ed398829ff"      # intent_tree_regrouping.md v1

def load_frozen_prompt(path, expected_hash, label):
    text = path.read_text(encoding="utf-8")
    actual = hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
    status = "OK" if actual == expected_hash else "MISMATCH"
    print(f"{label}: {path}\n  hash={actual}  expected={expected_hash}  [{status}]")
    if actual != expected_hash:
        print(f"  WARNING: {label} hash mismatch - comparison validity is void until reconciled.")
    return text

prompt_text = load_frozen_prompt(EXTRACTION_PROMPT_PATH, EXPECTED_EXTRACTION_HASH, "extraction prompt v1.1")
regroup_prompt_text = load_frozen_prompt(REGROUP_PROMPT_PATH, EXPECTED_REGROUP_HASH, "regrouping prompt v1")

In [ ]:
# Pick the ThoughtTrace conversation (the SAME id the frozen Sonnet extraction came from).
# Raw HF ids look like "user1016_task1_conversation1" (no prefix); the pipeline's simplified loader
# adds a "thoughttrace_" prefix, so we accept either form.
TARGET_CONV_ID = os.environ.get("TARGET_CONV_ID", "user1016_task1_conversation1")

tt = load_dataset("SCAI-JHU/ThoughtTrace", split="train", token=os.environ.get("HF_TOKEN"))
by_id = {rec["id"]: rec for rec in tt}

def find_record(conv_id):
    for cand in (conv_id, conv_id.removeprefix("thoughttrace_"), "thoughttrace_" + conv_id):
        if cand in by_id:
            return cand, by_id[cand]
    return None, None

resolved_id, record = find_record(TARGET_CONV_ID)
if record is None:
    raise KeyError(f"Set TARGET_CONV_ID to a valid id. Got {TARGET_CONV_ID!r}. "
                   f"Example ids: {list(by_id)[:3]}")

# Reproduce the frozen helper-prompt conversation format exactly, so Morpheus sees the same input
# the Claude Sonnet extraction did. Reasons inline on user turns; extraction instruction appended.
EXTRACT_INSTRUCTION = "Extract per the protocol and return only the JSON object."

def render_transcript(rec):
    lines = [f"CONVERSATION ID: {rec.get('id')}", "", "CONVERSATION:", ""]
    if rec.get("task_summary"):
        lines += ["[TASK_SUMMARY]", rec["task_summary"], ""]
    if rec.get("task_expectation"):
        lines += ["[TASK_EXPECTATION]", rec["task_expectation"], ""]
    for i, m in enumerate(rec.get("messages", []), start=1):
        speaker = "USER" if m.get("type") == "user" else "ASSISTANT"
        lines.append(f"[TURN {i} — {speaker}]")
        lines.append(m.get("content", ""))
        for r in (m.get("reasons") or []):
            lines.append(f"  (reason · {r.get('label')}): {r.get('content')}")
        lines.append("")
    return "\n".join(lines).rstrip()

transcript = render_transcript(record)
print(f"Resolved id: {resolved_id} - {len(record.get('messages', []))} messages\n")
print(transcript[:1500], "...")

In [ ]:
# (b) Extraction via Morpheus. SLOW: this single call can take many minutes - run interactively.
user_message = transcript + "\n\n" + EXTRACT_INSTRUCTION
raw = stream_chat(
    [{"role": "system", "content": prompt_text},
     {"role": "user", "content": user_message}],
    json_mode=True,
)
morpheus_tree = json.loads(raw)
print(json.dumps(morpheus_tree, indent=2)[:3000])

In [ ]:
# (b2) REGROUPING STEP - always run after extraction. A local analog of DiscoverLLM's stage-3
# hierarchy-organization (see the DiscoverLLM paper, arXiv:2602.03429): merge duplicate /
# near-duplicate intents, fix abstract->specific parent-child links, keep independent dimensions
# as separate trees, and enforce a DAG - without inventing or dropping intents.
def regroup(extraction_doc):
    payload = json.dumps(extraction_doc, ensure_ascii=False, indent=2)
    raw = stream_chat(
        [{"role": "system", "content": regroup_prompt_text},
         {"role": "user", "content": payload + "\n\nRegroup per the protocol and return only the JSON object."}],
        json_mode=True,
    )
    return json.loads(raw)

morpheus_regrouped = regroup(morpheus_tree)
print(json.dumps(morpheus_regrouped, indent=2)[:3000])

In [ ]:
# (c-prep) Frozen Sonnet extraction for the SAME conversation; structural stats for raw vs
# regrouped Morpheus vs Sonnet. Extraction schema: trees[] each {intent_type, nodes[]}; node =
# id/label/discovery_state/turn_of_discovery/parent_id/evidence (depth via parent_id chains).
SONNET_EXTRACTION_PATH = Path(os.environ.get(
    "SONNET_EXTRACTION_PATH", ROOT / "data" / "sonnet_extractions" / f"{TARGET_CONV_ID}.json"))
sonnet_tree = json.loads(SONNET_EXTRACTION_PATH.read_text(encoding="utf-8"))

def extraction_stats(doc):
    trees = doc.get("trees", [])
    nodes = [n for t in trees for n in t.get("nodes", [])]
    by_id = {n["id"]: n for n in nodes}
    def depth(n):
        d, cur = 1, n
        while cur.get("parent_id") is not None and cur.get("parent_id") in by_id:
            cur = by_id[cur["parent_id"]]
            d += 1
        return d
    return {
        "n_trees": len(trees),
        "n_nodes": len(nodes),
        "max_depth": max((depth(n) for n in nodes), default=0),
        "intent_types": sorted({t.get("intent_type") for t in trees}),
        "discovery_states": dict(Counter(n.get("discovery_state") for n in nodes)),
    }

comparison = pd.DataFrame({
    "morpheus_raw": extraction_stats(morpheus_tree),
    "morpheus_regrouped": extraction_stats(morpheus_regrouped),
    "claude_sonnet": extraction_stats(sonnet_tree),
})
comparison

In [ ]:
# (optional) Apply the SAME regrouping to the frozen Sonnet extraction (one more Morpheus call).
# Comment out to keep Morpheus calls to a minimum.
sonnet_regrouped = regroup(sonnet_tree)
print("Sonnet nodes raw -> regrouped:",
      extraction_stats(sonnet_tree)["n_nodes"], "->", extraction_stats(sonnet_regrouped)["n_nodes"])
print(json.dumps(sonnet_regrouped, indent=2)[:2500])

## (c) Qualitative comparison — structural, not a metric

> Valid **only** against the frozen extraction (`f1924f822892`) and regrouping (`33ed398829ff`) prompts.

From the table and printed trees, note (after running):

- **Regrouping effect:** how did raw → regrouped change node count, depth, and number of trees? Did it merge the right duplicates without dropping real intents?
- **Real-conversation vs. artifact trees:** how do these extracted trees compare to DiscoverLLM's artifact-derived ones (NB4)?
- **Intent-type coverage / discovery-state mix / schema discipline** — did Qwen emit out-of-enum values or malformed nodes (the rebuild notes flag enum slips on this model)?

Orientation for whether Morpheus/Qwen is usable for extraction; the manual Sonnet output is generally the better gold reference. **Not** a formal evaluation.